# 08 RAG-LLM Generation — Clinical Insights per Patient

Generates **doctor alerts**, **patient precautions**, and **follow-up recommendations** using Groq LLaMA-3.3-70B.

**Output format per patient:**
- `doctor_alert` — risk level + clinical summary for the attending physician
- `patient_precautions` — 4 actionable steps to prevent readmission
- `follow_up_recommendations` — care team follow-up guidance

In [10]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'groq', '-q'], check=True)
print('All packages ready.')

All packages ready.


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [11]:
import json, os, time
from groq import Groq

# ===================================================
# CONFIGURATION
# ===================================================
GROQ_API_KEY   = 'YOUR_GROQ_API_KEY_HERE'
MAX_PATIENTS   = 5    # change to 236 for full test-set run, or 1192 for all RAG patients

BASE_DIR       = '/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent'
PROMPTS_PATH   = os.path.join(BASE_DIR, 'rag_prompts.json')
OUTPUT_PATH    = os.path.join(BASE_DIR, 'llm_outputs_full.json')
# ===================================================

with open(PROMPTS_PATH) as f:
    rag_prompts = json.load(f)
print('Loaded', len(rag_prompts), 'prompts. Processing', MAX_PATIENTS)

SYSTEM_PROMPT = (
    'You are an expert clinical decision support AI embedded in a hospital readmission prevention system. '
    'You will receive a patient clinical summary, predicted 30-day readmission risk, and SHAP risk drivers. '
    'Respond ONLY with valid JSON using this exact structure: '
    '{"doctor_alert": {"risk_level": "HIGH / MEDIUM / LOW", '
    '"risk_summary": "2-3 sentence summary for the attending physician linking risk drivers to patient history."}, '
    '"patient_precautions": ['
    '"Precaution 1 written for the patient to understand.", '
    '"Precaution 2 written for the patient to understand.", '
    '"Precaution 3 written for the patient to understand.", '
    '"Precaution 4 written for the patient to understand."], '
    '"follow_up_recommendations": "1-2 sentence follow-up care recommendation for the care team."}. '
    'No markdown. No extra text. Only JSON.'
)

client = Groq(api_key=GROQ_API_KEY)

def generate_insight(prompt_context):
    response = client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt_context}
        ],
        temperature=0.3,
        response_format={'type': 'json_object'},
        max_tokens=1000
    )
    return response.choices[0].message.content

results = []
for i, item in enumerate(rag_prompts[:MAX_PATIENTS]):
    hid  = item['hadm_id']
    ctx  = item['prompt_context']
    risk = float(ctx.split('Predicted Readmission Risk:')[1].split('%')[0].strip()) if 'Predicted Readmission Risk:' in ctx else 0.0
    print('Patient', i+1, 'of', MAX_PATIENTS, '| hadm_id:', hid, end=' ... ')
    try:
        parsed = json.loads(generate_insight(ctx))
        results.append({
            'hadm_id':                   hid,
            'predicted_risk_pct':        round(risk, 1),
            'risk_level':                parsed.get('doctor_alert', {}).get('risk_level', ''),
            'doctor_alert':              parsed.get('doctor_alert', {}),
            'patient_precautions':       parsed.get('patient_precautions', []),
            'follow_up_recommendations': parsed.get('follow_up_recommendations', '')
        })
        print('OK')
    except Exception as e:
        results.append({'hadm_id': hid, 'predicted_risk_pct': round(risk, 1), 'error': str(e)})
        print('ERROR:', e)
    time.sleep(0.4)

with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved', len(results), 'results to llm_outputs_full.json')

Loaded 236 prompts. Processing 5
Patient 1 of 5 | hadm_id: 29158668 ... OK
Patient 2 of 5 | hadm_id: 23067520 ... ERROR: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kme5r6b9fb9t08nchwg6esbt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 100000, Requested 560. Please try again in 8m3.84s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Patient 3 of 5 | hadm_id: 27437666 ... ERROR: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kme5r6b9fb9t08nchwg6esbt` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99999, Requested 565. Please try again in 8m7.296s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Patient 4 of 5 | ha

In [12]:
# Load and display results
with open('/Users/sameer/Desktop/AIheathAgent/Capstone_Healthcare_Decision_Intelligence_Agent/llm_outputs_full.json') as f:
    all_results = json.load(f)

successful = [r for r in all_results if 'error' not in r]
print('Total records:', len(all_results), '| Successful:', len(successful))
print()

RISK_EMOJI = {'HIGH': 'CRITICAL', 'MEDIUM': 'MONITOR', 'LOW': 'STABLE'}

for r in successful:
    level = r.get('risk_level', 'UNKNOWN')
    label = RISK_EMOJI.get(level, level)
    da    = r.get('doctor_alert', {})

    print('=' * 65)
    print('PATIENT hadm_id:', r['hadm_id'],
          '| Predicted Risk:', str(r['predicted_risk_pct']) + '%',
          '|', level, '(' + label + ')')
    print('=' * 65)

    print()
    print('[DOCTOR ALERT]')
    print(' ', da.get('risk_summary', 'N/A'))

    print()
    print('[PATIENT PRECAUTIONS — To Avoid Readmission]')
    for idx, p in enumerate(r.get('patient_precautions', []), 1):
        print(' ', str(idx) + '.', p)

    print()
    print('[FOLLOW-UP RECOMMENDATIONS]')
    print(' ', r.get('follow_up_recommendations', 'N/A'))
    print()

Total records: 5 | Successful: 1

PATIENT hadm_id: 29158668 | Predicted Risk: 47.7% | HIGH (CRITICAL)

[DOCTOR ALERT]
  The patient is at high risk for readmission due to her complex medical history, including worsening heart failure, atrial fibrillation, and signs of fluid retention, as well as key risk drivers such as discharge to home health care, low hemoglobin levels, and marital status.

[PATIENT PRECAUTIONS — To Avoid Readmission]
  1. Monitor your weight daily and report any sudden increases to your healthcare provider.
  2. Take your medications as prescribed and do not skip doses.
  3. Follow a low-sodium diet to reduce fluid retention.
  4. Attend all scheduled follow-up appointments with your healthcare provider.

[FOLLOW-UP RECOMMENDATIONS]
  The care team should closely monitor the patient's condition, particularly her heart failure symptoms and hemoglobin levels, and consider adjusting her treatment plan as needed to prevent readmission.

